# Model 2 – Scalar Features

This notebook outputs a series of `slate_id`, `features: list<struct<...>>`. Each list element corresponds exactly to the same position in `candidate_wid_list`. Rows are streamed in physical input order, as per modelling decision done by the group.

## 1. Administrivia

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path
from collections import Counter
import math
import os
import pyarrow as pa
import pyarrow.parquet as pq

ROOT = Path('/content/drive/MyDrive/Grp8Submission/pipeline_data')
INPUTS = {
    'train': ROOT / 'step1/train_sampled.parquet',
    'val1': ROOT / 'step1/val1.parquet',
    'val2': ROOT / 'step1/val2.parquet',
    'test': ROOT / 'step1/test.parquet',
}
# Accept the spelling in the request if that is the actual filename.
if not INPUTS['val2'].exists() and (ROOT / 'step1/val2.paruqet').exists():
    INPUTS['val2'] = ROOT / 'step1/val2.paruqet'
METADATA_PATH = ROOT / 'step1/product_metadata_clean.parquet'
OUTPUT_DIR = ROOT / 'step2_model2_scalar'
BATCH_SIZE = 512
OVERWRITE = True
REQUIRED = ['slate_id', 'candidate_wid_list', 'history_wid_list',
            'history_type_list', 'history_days_before']
EVENTS = {
    'CLICK': ('click', 7., 1),
    'FLW': ('cart', 90., 2),
    'CART': ('cart', 90., 5),
    'ORD': ('order', 365., 10),
}
SCOPES = ('all', 'click', 'cart', 'order')


## 2. Helper Functions

The following features are engineered

| Feature | Description | Remarks |
|---|---|---|
| `<ch>_brand_affinity_ratio` | Current candidate’s brand affinity normalised over total brand affinity | 0 if the candidate’s brand is not found or the slate has no history |
| `<ch>_cate_affinity_ratio` | Current candidate’s L4 category affinity normalised over total category affinity | 0 if the candidate’s category is not found or the slate has no history |
| `<ch>_is_dominant_brand` | Binary indicator of whether the current candidate’s brand is the dominant historical brand | All 0 if the slate has no history |
| `<ch>_hist_item_count` | Number of historical interactions in the channel | 0 if the slate has no history |

In [ ]:
def missing(value):
    """
    Return True if value is None or NaN.
    """
    return value is None or (isinstance(value, float) and math.isnan(value))


def clean_id(value):
    """
    Returns ID properly formatted as an integer or None
    """
    if missing(value) or (isinstance(value, str) and not value.strip()):
        return None
    number = int(value)
    if isinstance(value, float) and value != number:
        raise ValueError(f'Noninteger ID: {value!r}')
    return number if number >= 0 else None


def load_metadata(path):
    """
    Loads metadata from METADATA_PATH
    """
    expected = {'wid', 'brand_id', 'cate_id_1', 'cate_id_2',
                'cate_id_3', 'cate_id_4', 'shop_id'}
    pf = pq.ParquetFile(path)
    if not expected.issubset(pf.schema_arrow.names):
        raise ValueError(f'Metadata missing columns: {expected-set(pf.schema_arrow.names)}')
    lookup = {}
    for batch in pf.iter_batches(batch_size=65536, columns=['wid', 'brand_id', 'cate_id_4']):
        for row in batch.to_pylist():
            wid = clean_id(row['wid'])
            if wid is None:
                raise ValueError('Missing metadata wid')
            attrs = (clean_id(row['brand_id']), clean_id(row['cate_id_4']))
            if wid in lookup and lookup[wid] != attrs:
                raise ValueError(f'Conflicting metadata for wid={wid}')
            lookup[wid] = attrs
    return lookup


def scope_stats(events, metadata):
    """
    Process all historical events and return the number of historical data,
    dominant brands, total weight, and brand and category affinity weights.
    """
    brands, brand_weights, cate_weights = Counter(), Counter(), Counter()
    total = 0
    for wid, weight in events:
        brand, cate = metadata.get(clean_id(wid), (None, None))
        total += weight
        if brand is not None:
            brands[brand] += 1
            brand_weights[brand] += weight
        if cate is not None:
            cate_weights[cate] += weight
    dominant = max(brands, key=brands.get) if brands else float("nan")
    return len(events), dominant, total, brand_weights, cate_weights


def ratio(candidate_attr, weights):
    """
    Safe ratio (in case numerator is missing or denominator is missing or zero)
    """
    if candidate_attr is None:
        return 0

    known_total = sum(weights.values())
    if known_total == 0:
        return 0

    return weights.get(candidate_attr, 0) / known_total


def engineer_row(row, metadata):
    """
    Main feature engineering helper function
    """

    # Sanity checks and NA-handling
    candidates = row['candidate_wid_list']
    if candidates is None:
        raise ValueError('candidate_wid_list is missing; candidate alignment is unavailable')
    history = row['history_wid_list']
    if history is None:
        history, types, ages = [], [], []
    else:
        types = row["history_type_list"]
        ages = row["history_days_before"]
    if types is None or ages is None or not (len(history) == len(types) == len(ages)):
        raise ValueError('History wid/type/age lists are missing or misaligned')

    # Process historical events
    events = {scope: [] for scope in SCOPES}
    for wid, event_type, age in zip(history, types, ages):
        if not isinstance(event_type, str) or event_type not in EVENTS:
            raise ValueError(f'Unknown/missing event type: {event_type!r}')
        channel, window, weight = EVENTS[event_type]
        if age is None or not math.isfinite(float(age)) or float(age) < 0:
            raise ValueError(f'Invalid history_days_before: {age!r}')
        if float(age) <= window: # Filtering for the correct event period
            event = (wid, weight)
            events['all'].append(event)
            events[channel].append(event)
    stats = {scope: scope_stats(events[scope], metadata) for scope in SCOPES}

    # Processing results from ratio into desired features
    result = []
    for candidate in candidates:
        brand, cate = metadata.get(clean_id(candidate), (None, None))
        features = {}
        for scope in SCOPES:
            suffix = '' if scope == 'all' else scope + '_'
            count, dominant, total, bw, cw = stats[scope]
            features[suffix + "is_dominant_brand"] = int(dominant == brand)
            features[suffix + 'brand_affinity_ratio'] = ratio(brand, bw)
            features[suffix + 'cate_affinity_ratio'] = ratio(cate, cw)
            features[suffix + 'hist_item_count'] = count
        result.append(features)
    return result


FEATURE_NAMES = []
for scope in SCOPES:
    suffix = '' if scope == 'all' else scope + '_'
    FEATURE_NAMES.extend(suffix + base for base in
                         ('is_dominant_brand', 'brand_affinity_ratio',
                          'cate_affinity_ratio', 'hist_item_count'))


### 2.1. Sanity Check

To verify whether helper funtions work as intended

In [ ]:
meta = {1: (10, 100), 2: (20, 200), 3: (30, 300)}
row = dict(slate_id=9, candidate_wid_list=[1, 2, 3, 1],
           history_wid_list=[1, 2, 1, 2, 3],
           history_type_list=['CLICK', 'FLW', 'CART', 'ORD', 'CLICK'],
           history_days_before=[7, 90, 90, 365, 7.00001])
f = engineer_row(row, meta)
assert len(f) == 4 and f[0] == f[3]
assert f[0]['hist_item_count'] == 4
assert f[0]["is_dominant_brand"] == 1
assert math.isclose(f[0]['brand_affinity_ratio'], 6/18)
assert math.isclose(f[1]['cate_affinity_ratio'], 12/18)
assert math.isclose(f[0]['cart_brand_affinity_ratio'], 5/7)
assert f[2]['brand_affinity_ratio'] == 0
empty = dict(row, history_wid_list=[], history_type_list=[], history_days_before=[])
e = engineer_row(empty, meta)[0]
assert e["hist_item_count"] == 0
assert e["brand_affinity_ratio"] == 0
assert e["is_dominant_brand"] == 0
assert (
    engineer_row(dict(row, history_wid_list=None), meta)[0]["hist_item_count"]
    == 0
)
assert engineer_row(dict(row, candidate_wid_list=[999]), meta)[0][
        "brand_affinity_ratio"
    ] == 0
partial = engineer_row(row, {1: (10, 100), 2: (None, 200)})[0]
assert math.isclose(partial['brand_affinity_ratio'], 1.0)
assert math.isclose(partial['cate_affinity_ratio'], 6/18)
assert partial['hist_item_count'] == 4
for event, boundary in [('CLICK', 7), ('FLW', 90), ('CART', 90), ('ORD', 365)]:
    r = dict(row, history_wid_list=[1, 1], history_type_list=[event, event],
             history_days_before=[boundary, boundary + 0.00001])
    assert engineer_row(r, meta)[0]['hist_item_count'] == 1
for changes in [dict(history_days_before=[0]),
                dict(history_days_before=[-1]*5),
                dict(history_type_list=['UNKNOWN']*5),
                dict(history_days_before=[None]*5)]:
    try:
        engineer_row(dict(row, **changes), meta)
    except ValueError:
        pass
    else:
        raise AssertionError('Invalid history was silently accepted')
print('Feature checks passed')


Feature checks passed


### 2.2. Input/Output Check

Defining output schema and trying to output a temporary file as a test.

In [ ]:
FEATURE_TYPE = pa.struct([
    pa.field(
        name,
        pa.int64() if name.endswith(("hist_item_count", "is_dominant_brand")) else pa.float64(),
    )
    for name in FEATURE_NAMES
])


def output_schema(slate_type):
    return pa.schema([pa.field('slate_id', slate_type),
                      pa.field('features', pa.list_(FEATURE_TYPE))],
                     metadata={b'candidate_alignment': b'features[i] matches candidate_wid_list[i]',
                               b'feature_rules': b'CLICK<=7;FLW/CART<=90;ORD<=365;inclusive;FLW in cart'})


def make_output(rows, schema, metadata):
    payload = []
    for row in rows:
        try:
            payload.append(engineer_row(row, metadata))
        except Exception as exc:
            raise ValueError(f"slate_id={row['slate_id']!r}: {exc}") from exc
    return pa.Table.from_arrays([
        pa.array([r['slate_id'] for r in rows], type=schema.field('slate_id').type),
        pa.array(payload, type=schema.field('features').type, from_pandas=False),
    ], schema=schema)

import tempfile
with tempfile.TemporaryDirectory() as d:
    p = Path(d) / 'check.parquet'
    test_rows = [dict(empty, slate_id=9), dict(row, slate_id=2)]
    table = make_output(test_rows, output_schema(pa.int32()), meta)
    pq.write_table(table, p)
    back = pq.read_table(p).to_pylist()
    assert [r['slate_id'] for r in back] == [9, 2]
    assert back[0]["features"][0]["brand_affinity_ratio"] == 0
    assert len(back[1]['features']) == len(row['candidate_wid_list'])
print('Parquet round-trip checks passed')


Parquet round-trip checks passed


## 3. Split Processing

Processes various splits (train/val1/val2/test)

In [ ]:
def process_split(source, destination, metadata):
    # Check file existence
    source, destination = Path(source), Path(destination)
    if destination.exists() and not OVERWRITE:
        raise FileExistsError(f'{destination}; set OVERWRITE=True to replace')
    pf = pq.ParquetFile(source)
    absent = set(REQUIRED) - set(pf.schema_arrow.names)

    # Input validation and output schema development
    if absent:
        raise ValueError(f'{source}: missing columns {absent}')
    schema = output_schema(pf.schema_arrow.field('slate_id').type)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix('.parquet.partial')

    # Write output along with slate_id for row matching in downstream tasks
    count = 0
    try:
        with pq.ParquetWriter(temporary, schema, compression='zstd') as writer:
            for batch in pf.iter_batches(batch_size=BATCH_SIZE, columns=REQUIRED, use_threads=False):
                rows = batch.to_pylist()
                table = make_output(rows, schema, metadata)
                writer.write_table(table)
                count += len(rows)
        assert count == pf.metadata.num_rows

        # Stream IDs to verify physical row alignment against the original file.
        out_pf = pq.ParquetFile(temporary)
        src_ids = (v for b in pf.iter_batches(batch_size=4096, columns=['slate_id'])
                   for v in b.column(0).to_pylist())
        out_ids = (v for b in out_pf.iter_batches(batch_size=4096, columns=['slate_id'])
                   for v in b.column(0).to_pylist())
        from itertools import zip_longest
        sentinel = object()
        assert all(a == b for a, b in zip_longest(src_ids, out_ids, fillvalue=sentinel))
        assert out_pf.metadata.num_rows == count
        os.replace(temporary, destination)
    except BaseException:
        temporary.unlink(missing_ok=True)
        raise
    print(f'{source.name}: {count:,} slates -> {destination}')
    return destination


# Check all paths before writing any output.
for path in [METADATA_PATH, *INPUTS.values()]:
    if not path.is_file():
        raise FileNotFoundError(f'{path}; adjust ROOT / INPUTS above')
for split in INPUTS:
    target = OUTPUT_DIR / f'{split}_scalar_features.parquet'
    if target.exists() and not OVERWRITE:
        raise FileExistsError(target)
metadata = load_metadata(METADATA_PATH)
if any(brand is not None and brand > 2**53 for brand, _ in metadata.values()):
    raise ValueError('brand_id exceeds exact float64 integer range')
outputs = {split: process_split(path, OUTPUT_DIR / f'scalar_{split}.parquet', metadata)
           for split, path in INPUTS.items()}


train_sampled.parquet: 122,083 slates -> /content/drive/MyDrive/Grp8Submission/pipeline_data/step2_model2_scalar/scalar_train.parquet
val1.parquet: 17,799 slates -> /content/drive/MyDrive/Grp8Submission/pipeline_data/step2_model2_scalar/scalar_val1.parquet
val2.parquet: 16,924 slates -> /content/drive/MyDrive/Grp8Submission/pipeline_data/step2_model2_scalar/scalar_val2.parquet
test.parquet: 17,025 slates -> /content/drive/MyDrive/Grp8Submission/pipeline_data/step2_model2_scalar/scalar_test.parquet


### 3.1. Example

The following code shows how an example slate looks like (Exploded by candidate wid for easier viewing)

In [ ]:
import pandas as pd
import pyarrow.parquet as pq

# Read the first 10 slates.
pf = pq.ParquetFile(outputs['val1'])
preview = next(pf.iter_batches(batch_size=10)).to_pylist()

# One row per candidate; retain the corresponding slate_id.
df_preview = pd.json_normalize(
    preview,
    record_path='features',
    meta=['slate_id']
)

display(df_preview[df_preview['slate_id'] == 18])

,is_dominant_brand,brand_affinity_ratio,cate_affinity_ratio,hist_item_count,click_is_dominant_brand,click_brand_affinity_ratio,click_cate_affinity_ratio,click_hist_item_count,cart_is_dominant_brand,cart_brand_affinity_ratio,cart_cate_affinity_ratio,cart_hist_item_count,order_is_dominant_brand,order_brand_affinity_ratio,order_cate_affinity_ratio,order_hist_item_count,slate_id
0,1,0.20143,0.195471,234,1,0.734043,0.734043,94,1,0.165289,0.157025,123,0,0.0,0.0,17,18
1,1,0.20143,0.195471,234,1,0.734043,0.734043,94,1,0.165289,0.157025,123,0,0.0,0.0,17,18
2,1,0.20143,0.000000,234,1,0.734043,0.000000,94,1,0.165289,0.000000,123,0,0.0,0.0,17,18
3,1,0.20143,0.195471,234,1,0.734043,0.734043,94,1,0.165289,0.157025,123,0,0.0,0.0,17,18
4,1,0.20143,0.195471,234,1,0.734043,0.734043,94,1,0.165289,0.157025,123,0,0.0,0.0,17,18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87,1,0.20143,0.195471,234,1,0.734043,0.734043,94,1,0.165289,0.157025,123,0,0.0,0.0,17,18
88,1,0.20143,0.195471,234,1,0.734043,0.734043,94,1,0.165289,0.157025,123,0,0.0,0.0,17,18
89,1,0.20143,0.195471,234,1,0.734043,0.734043,94,1,0.165289,0.157025,123,0,0.0,0.0,17,18
90,1,0.20143,0.195471,234,1,0.734043,0.734043,94,1,0.165289,0.157025,123,0,0.0,0.0,17,18


### 3.2. Missingness

Find percent of missing entries in train/val1/val2/test split. By right there should be no missing values as missingness are already handled in this notebook.

In [ ]:
import pandas as pd
import pyarrow.parquet as pq

summaries = []

for split in ("train", "val1", "val2", "test"):
    pf = pq.ParquetFile(OUTPUT_DIR / f"scalar_{split}.parquet")

    # Extract feature names from the nested struct schema.
    struct_type = pf.schema_arrow.field("features").type.value_type
    feature_cols = [
        field.name for field in struct_type
        if field.name not in {"wid", "candidate_pos"}
    ]

    missing_counts = pd.Series(0, index=feature_cols, dtype="int64")
    total_candidates = 0

    for batch in pf.iter_batches(batch_size=128, columns=["features"]):
        records = [
            record
            for slate in batch.to_pylist()
            for record in (slate["features"] or [])
        ]
        if not records:
            continue

        df = pd.DataFrame(records, columns=feature_cols)

        missing_counts += (df.isna() | df.eq(-1)).sum()
        total_candidates += len(df)

    summary = (
        missing_counts
        .rename_axis("feature")
        .reset_index(name="missing_count")
    )
    summary["split"] = split
    summary["total_candidates"] = total_candidates
    summary["missing_proportion"] = (
        summary["missing_count"] / total_candidates
        if total_candidates else float("nan")
    )
    summaries.append(summary)

missingness = pd.concat(summaries, ignore_index=True)

missingness_table = (
    missingness.pivot(
        index="feature",
        columns="split",
        values="missing_proportion",
    )
    .reindex(columns=["train", "val1", "val2", "test"])
    .sort_values("train", ascending=False)
)

with pd.option_context("display.max_rows", None):
    display(missingness_table.style.format("{:.2%}"))

split,train,val1,val2,test
feature,,,,
brand_affinity_ratio,0.00%,0.00%,0.00%,0.00%
cart_brand_affinity_ratio,0.00%,0.00%,0.00%,0.00%
cart_cate_affinity_ratio,0.00%,0.00%,0.00%,0.00%
cart_hist_item_count,0.00%,0.00%,0.00%,0.00%
cart_is_dominant_brand,0.00%,0.00%,0.00%,0.00%
cate_affinity_ratio,0.00%,0.00%,0.00%,0.00%
click_brand_affinity_ratio,0.00%,0.00%,0.00%,0.00%
click_cate_affinity_ratio,0.00%,0.00%,0.00%,0.00%
click_hist_item_count,0.00%,0.00%,0.00%,0.00%


## 4. Outputs

`step2_model2_scalar/scalar_train.parquet`, `step2_model2_scalar/scalar_val1.parquet`, `step2_model2_scalar/scalar_val2.parquet`, `step2_model2_scalar/scalar_test.parquet`.

Each candidate record has 16 scalar features: four features for each of pooled/click/cart/order scopes. Slate-level quantities repeat across candidates so all Model 2 inputs remain in a single candidate-aligned record.